# 05 - Skin Patch Classification & Domain Adaptation

Trains an EfficientNet-B0 classifier on ACNE04 patches, evaluates cross-domain transfer to DermNet, and applies multiple domain adaptation strategies (histogram matching, Reinhard normalization, few-shot fine-tuning).

## Setup

In [ ]:
import os
import json
import getpass
from pathlib import Path

if not os.path.exists("/kaggle/working/yanglab-acne"):
    !git clone https://github.com/kaizar-rang/yanglab-acne.git
%cd /kaggle/working/yanglab-acne

%pip install roboflow kaggle grad-cam -q
print("\nDone.")

## Credentials & Dataset Download

In [ ]:
roboflow_key = getpass.getpass("Enter ROBOFLOW_API_KEY: ")
kaggle_username = getpass.getpass("Enter KAGGLE_USERNAME: ")
kaggle_key = getpass.getpass("Enter KAGGLE_KEY: ")

os.environ["ROBOFLOW_API_KEY"] = roboflow_key
os.environ["KAGGLE_USERNAME"] = kaggle_username
os.environ["KAGGLE_KEY"] = kaggle_key

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
with open(kaggle_dir / "kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(kaggle_dir / "kaggle.json", 0o600)

with open(".env", "w") as f:
    f.write(f"ROBOFLOW_API_KEY={roboflow_key}\n")
    f.write(f"KAGGLE_USERNAME={kaggle_username}\n")
    f.write(f"KAGGLE_KEY={kaggle_key}\n")

!echo "4" | python setup.py
print("\nSetup complete.")

In [ ]:
!python src/data/patch_extractor.py

## Environment Check

In [ ]:
import torch
import torchvision

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

from pathlib import Path
patch_dir = Path("data/patches")
train_acne = len(list((patch_dir / "train" / "acne").glob("*.jpg")))
train_clear = len(list((patch_dir / "train" / "clear").glob("*.jpg")))
print(f"\nTrain patches -- acne: {train_acne}, clear: {train_clear}")
print(f"Ratio: {train_acne/train_clear:.2f}:1")

dermnet_test = Path("data/dermnet/test")
conditions = [d.name for d in dermnet_test.iterdir() if d.is_dir()]
print(f"\nDermNet test conditions: {len(conditions)}")

## Part A: Baseline EfficientNet-B0 on ACNE04 Patches

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder("data/patches/train", transform=train_transforms)
val_dataset = ImageFolder("data/patches/val", transform=val_transforms)
print("Class to idx:", train_dataset.class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

def build_classifier(num_classes=2):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_classifier(num_classes=2).to(device)

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

class_counts = [len(list((Path("data/patches/train/acne")).glob("*.jpg"))),
                len(list((Path("data/patches/train/clear")).glob("*.jpg")))]
total = sum(class_counts)
weights = torch.tensor([total/c for c in class_counts]).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=20)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

os.makedirs("outputs/checkpoints/classifier", exist_ok=True)

num_epochs = 20
best_val_acc = 0
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, device)
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "outputs/checkpoints/classifier/best.pt")

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f}")

print(f"\nBest val accuracy: {best_val_acc:.3f}")

### ACNE04 Validation Metrics (Accuracy, F1, AUROC)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

model.load_state_dict(torch.load("outputs/checkpoints/classifier/best.pt"))
model.eval()

all_preds_val, all_labels_val, all_probs_val = [], [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = outputs.argmax(1)
        all_preds_val.extend(preds.cpu().numpy())
        all_labels_val.extend(labels.numpy())
        all_probs_val.extend(probs.cpu().numpy())

acc_val = accuracy_score(all_labels_val, all_preds_val)
f1_val = f1_score(all_labels_val, all_preds_val)
auroc_val = roc_auc_score(all_labels_val, all_probs_val)

print("ACNE04 Val Results:")
print(f"  Accuracy: {acc_val:.3f}")
print(f"  F1:       {f1_val:.3f}")
print(f"  AUROC:    {auroc_val:.3f}")

## Part B: Baseline Evaluation on DermNet (No Adaptation)

**Note:** ACNE04 patches use `class_to_idx = {'acne': 0, 'clear': 1}` (alphabetical, via ImageFolder). The DermNet dataset below is constructed with the same convention: acne = 0, clear = 1.

In [ ]:
from PIL import Image

class ACNEBinaryDataset(torch.utils.data.Dataset):
    """label 0 = acne, 1 = clear -- matches ImageFolder's alphabetical class_to_idx for ACNE04 patches"""
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for condition_dir in sorted(Path(root_dir).iterdir()):
            if not condition_dir.is_dir():
                continue
            label = 0 if "acne" in condition_dir.name.lower() else 1
            for img_path in condition_dir.glob("*.jpg"):
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

dermnet_dataset = ACNEBinaryDataset("data/dermnet/test", transform=val_transforms)
dermnet_loader = DataLoader(dermnet_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"DermNet test samples: {len(dermnet_dataset)}")
n_acne = sum(1 for _, l in dermnet_dataset.samples if l == 0)
print(f"Acne: {n_acne}, Non-acne: {len(dermnet_dataset) - n_acne}, prevalence: {n_acne/len(dermnet_dataset):.3f}")

In [ ]:
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in dermnet_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 0]  # P(acne), acne = class 0
        preds = outputs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

# Flip for sklearn convention (positive class = acne = 0 -> treat as 1 for metrics)
labels_pos = [1 - l for l in all_labels]
preds_pos = [1 - p for p in all_preds]

acc = accuracy_score(labels_pos, preds_pos)
f1 = f1_score(labels_pos, preds_pos)
auroc = roc_auc_score(labels_pos, all_probs)

print("Baseline DermNet Results (no adaptation):")
print(f"  Accuracy: {acc:.3f}")
print(f"  F1:       {f1:.3f}")
print(f"  AUROC:    {auroc:.3f}")

## Part C: Domain Adaptation

Three strategies applied: (1) histogram matching, (2) Reinhard color normalization in LAB space, (3) few-shot fine-tuning on 20 unlabeled DermNet training samples with strong augmentation.

In [ ]:
import numpy as np
from PIL import Image
import cv2
import random

def histogram_match(source, reference):
    source = np.array(source).astype(np.float32)
    reference = np.array(reference).astype(np.float32)
    matched = np.zeros_like(source)
    for channel in range(3):
        src_hist, _ = np.histogram(source[:, :, channel].flatten(), 256, [0, 256])
        ref_hist, _ = np.histogram(reference[:, :, channel].flatten(), 256, [0, 256])
        src_cdf = src_hist.cumsum() / src_hist.sum()
        ref_cdf = ref_hist.cumsum() / ref_hist.sum()
        lookup = np.zeros(256)
        ref_idx = 0
        for src_idx in range(256):
            while ref_idx < 255 and ref_cdf[ref_idx] < src_cdf[src_idx]:
                ref_idx += 1
            lookup[src_idx] = ref_idx
        matched[:, :, channel] = lookup[source[:, :, channel].astype(np.uint8)]
    return Image.fromarray(matched.clip(0, 255).astype(np.uint8))

def reinhard_normalize(source, reference):
    source = np.array(source).astype(np.float32)
    reference = np.array(reference).astype(np.float32)
    src_lab = cv2.cvtColor(source / 255.0, cv2.COLOR_RGB2LAB)
    ref_lab = cv2.cvtColor(reference / 255.0, cv2.COLOR_RGB2LAB)
    for c in range(3):
        src_lab[:, :, c] = (src_lab[:, :, c] - src_lab[:, :, c].mean()) / (src_lab[:, :, c].std() + 1e-6)
        src_lab[:, :, c] = src_lab[:, :, c] * ref_lab[:, :, c].std() + ref_lab[:, :, c].mean()
    result = cv2.cvtColor(np.clip(src_lab, 0, 255).astype(np.float32), cv2.COLOR_LAB2RGB)
    return Image.fromarray((result * 255).clip(0, 255).astype(np.uint8))

### C1: Histogram Matching Only (Baseline Adaptation)

In [ ]:
dermnet_train_dir = Path("data/dermnet/train")
reference_images = []
for condition_dir in sorted(dermnet_train_dir.iterdir()):
    if condition_dir.is_dir():
        imgs = list(condition_dir.glob("*.jpg"))
        if imgs:
            reference_images.append(Image.open(random.choice(imgs)).convert("RGB"))

reference_img = reference_images[0]
print(f"Reference images available: {len(reference_images)}")
print("Using DermNet reference image for histogram matching")

class HistMatchDermNetDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, reference_img, transform=None):
        self.samples = []
        self.reference_img = reference_img
        self.transform = transform
        for condition_dir in sorted(Path(root_dir).iterdir()):
            if not condition_dir.is_dir():
                continue
            label = 0 if "acne" in condition_dir.name.lower() else 1
            for img_path in condition_dir.glob("*.jpg"):
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        img = histogram_match(img, self.reference_img)
        if self.transform:
            img = self.transform(img)
        return img, label

hist_test_dataset = HistMatchDermNetDataset("data/dermnet/test", reference_img, transform=val_transforms)
hist_test_loader = DataLoader(hist_test_dataset, batch_size=64, shuffle=False, num_workers=0)

all_preds_hist, all_labels_hist, all_probs_hist = [], [], []
with torch.no_grad():
    for images, labels in hist_test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 0]
        preds = outputs.argmax(1)
        all_preds_hist.extend(preds.cpu().numpy())
        all_labels_hist.extend(labels.numpy())
        all_probs_hist.extend(probs.cpu().numpy())

labels_pos_h = [1 - l for l in all_labels_hist]
preds_pos_h = [1 - p for p in all_preds_hist]

acc_adapted = accuracy_score(labels_pos_h, preds_pos_h)
f1_adapted = f1_score(labels_pos_h, preds_pos_h)
auroc_adapted = roc_auc_score(labels_pos_h, all_probs_hist)

print("Histogram Matching Results:")
print(f"  Accuracy: {acc_adapted:.3f}")
print(f"  F1:       {f1_adapted:.3f}")
print(f"  AUROC:    {auroc_adapted:.3f}")

### C2 + C3: Reinhard Normalization + Few-Shot Fine-Tuning

Freeze backbone except last block + head; fine-tune on 20 DermNet training samples for 30 epochs with strong augmentation, applying histogram matching + Reinhard normalization to all images.

In [ ]:
import torchvision.transforms.functional as TF

os.makedirs("outputs/checkpoints/classifier_adapted2", exist_ok=True)

# 1. Build 20-sample few-shot set from DermNet training data
few_shot_samples = []
for condition_dir in sorted(dermnet_train_dir.iterdir()):
    if not condition_dir.is_dir():
        continue
    label = 0 if "acne" in condition_dir.name.lower() else 1
    imgs = list(condition_dir.glob("*.jpg"))
    if imgs:
        img = Image.open(random.choice(imgs)).convert("RGB")
        few_shot_samples.append((img, label))

few_shot_samples = random.sample(few_shot_samples, min(20, len(few_shot_samples)))
print(f"Few-shot samples: {len(few_shot_samples)}")

# 2. Few-shot dataset with histogram match + Reinhard + strong augmentation
few_shot_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.GaussianBlur(kernel_size=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class FewShotDataset(torch.utils.data.Dataset):
    def __init__(self, samples, reference_img, transform=None):
        self.samples = samples
        self.reference_img = reference_img
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img, label = self.samples[idx]
        img = histogram_match(img, self.reference_img)
        img = reinhard_normalize(img, self.reference_img)
        if self.transform:
            img = self.transform(img)
        return img, label

few_shot_dataset = FewShotDataset(few_shot_samples, reference_img, transform=few_shot_transforms)
few_shot_loader = DataLoader(few_shot_dataset, batch_size=4, shuffle=True, num_workers=0)

# 3. Build adapted model -- freeze backbone, unfreeze last block + head
model_adapted = build_classifier(num_classes=2)
model_adapted.load_state_dict(torch.load("outputs/checkpoints/classifier/best.pt"))
model_adapted = model_adapted.to(device)

for param in model_adapted.parameters():
    param.requires_grad = False
for param in model_adapted.features[-1].parameters():
    param.requires_grad = True
for param in model_adapted.classifier.parameters():
    param.requires_grad = True

optimizer_adapted = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_adapted.parameters()),
    lr=1e-4, weight_decay=1e-4
)
criterion_adapted = nn.CrossEntropyLoss()

# 4. Few-shot fine-tuning
print("Fine-tuning on 20 DermNet samples...")
model_adapted.train()
for epoch in range(30):
    for images, labels in few_shot_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_adapted.zero_grad()
        outputs = model_adapted(images)
        loss = criterion_adapted(outputs, labels)
        loss.backward()
        optimizer_adapted.step()
    if (epoch + 1) % 10 == 0:
        print(f"  Fine-tune epoch {epoch+1}/30 | Loss: {loss.item():.4f}")

torch.save(model_adapted.state_dict(), "outputs/checkpoints/classifier_adapted2/best.pt")
print("Saved adapted model.")

In [ ]:
# 5. Evaluate fully adapted model on DermNet test set
val_transforms_adapted = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class AdaptedDermNetDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, reference_img, transform=None):
        self.samples = []
        self.reference_img = reference_img
        self.transform = transform
        for condition_dir in sorted(Path(root_dir).iterdir()):
            if not condition_dir.is_dir():
                continue
            label = 0 if "acne" in condition_dir.name.lower() else 1
            for img_path in condition_dir.glob("*.jpg"):
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        img = histogram_match(img, self.reference_img)
        img = reinhard_normalize(img, self.reference_img)
        if self.transform:
            img = self.transform(img)
        return img, label

adapted_test_dataset = AdaptedDermNetDataset("data/dermnet/test", reference_img, transform=val_transforms_adapted)
adapted_test_loader = DataLoader(adapted_test_dataset, batch_size=64, shuffle=False, num_workers=0)

model_adapted.eval()
all_preds2, all_labels2, all_probs2 = [], [], []
with torch.no_grad():
    for images, labels in adapted_test_loader:
        images = images.to(device)
        outputs = model_adapted(images)
        probs = torch.softmax(outputs, dim=1)[:, 0]
        preds = outputs.argmax(1)
        all_preds2.extend(preds.cpu().numpy())
        all_labels2.extend(labels.numpy())
        all_probs2.extend(probs.cpu().numpy())

labels_pos2 = [1 - l for l in all_labels2]
preds_pos2 = [1 - p for p in all_preds2]

acc2 = accuracy_score(labels_pos2, preds_pos2)
f1_2 = f1_score(labels_pos2, preds_pos2)
auroc2 = roc_auc_score(labels_pos2, all_probs2)

print("\nFull Domain Adaptation Comparison:")
print(f"{'Metric':<12} {'Baseline':>10} {'Hist Match':>12} {'Full Adapt':>12}")
print("-" * 50)
print(f"{'Accuracy':<12} {acc:.3f}{'':>7} {acc_adapted:.3f}{'':>9} {acc2:.3f}")
print(f"{'F1':<12} {f1:.3f}{'':>7} {f1_adapted:.3f}{'':>9} {f1_2:.3f}")
print(f"{'AUROC':<12} {auroc:.3f}{'':>7} {auroc_adapted:.3f}{'':>9} {auroc2:.3f}")

### Threshold Sweep on Fully Adapted Model

In [ ]:
for threshold in [0.05, 0.08, 0.10, 0.15]:
    preds_t = (np.array(all_probs2) >= threshold).astype(int)
    print(f"Threshold {threshold:.2f} -- Acc: {accuracy_score(labels_pos2, preds_t):.3f} "
          f"F1: {f1_score(labels_pos2, preds_t):.3f}")

## Part D: Grad-CAM Visualizations on DermNet (Adapted Model)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt

os.makedirs("outputs/predictions", exist_ok=True)

ACNE_CLASS_IDX = 0  # acne = 0 (alphabetical ImageFolder convention)
class_names = {0: "acne", 1: "clear"}

acne_indices = [i for i, (_, l) in enumerate(adapted_test_dataset.samples) if l == 0]
clear_indices = [i for i, (_, l) in enumerate(adapted_test_dataset.samples) if l == 1]
sample_indices = random.sample(acne_indices, 5) + random.sample(clear_indices, 5)

target_layers = [model_adapted.features[-1]]
fig, axes = plt.subplots(2, 10, figsize=(25, 6))

with GradCAM(model=model_adapted, target_layers=target_layers) as cam:
    for i, idx in enumerate(sample_indices):
        img_tensor, true_label = adapted_test_dataset[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model_adapted(input_tensor)
            pred_idx = output.argmax(1).item()
            prob = torch.softmax(output, dim=1)[0, ACNE_CLASS_IDX].item()

        grayscale_cam = cam(input_tensor=input_tensor,
                             targets=[ClassifierOutputTarget(ACNE_CLASS_IDX)])[0]

        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = (img_np * std + mean).clip(0, 1).astype(np.float32)

        visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        axes[0, i].imshow(img_np)
        axes[0, i].set_title(f"True: {class_names[true_label]}", fontsize=8)
        axes[0, i].axis("off")

        axes[1, i].imshow(visualization)
        axes[1, i].set_title(f"Pred: {class_names[pred_idx]} ({prob:.2f})", fontsize=8)
        axes[1, i].axis("off")

plt.suptitle("Grad-CAM Visualizations on DermNet Test Set (Adapted Model)", fontsize=13)
plt.tight_layout()
plt.savefig("outputs/predictions/gradcam_dermnet.png", dpi=150)
plt.show()
print("Saved to outputs/predictions/gradcam_dermnet.png")

## Save & Download Results

In [ ]:
!zip -r classification_results.zip \
    outputs/checkpoints/classifier/best.pt \
    outputs/checkpoints/classifier_adapted2/best.pt \
    outputs/predictions/gradcam_dermnet.png

print("Saved classification_results.zip")